# Main file 

05.11.2025

Written by: Arnav Kumar Upadhyay, Cedric Meyer adapted from MATLAB code of Marc, Jan Spindler DOI-https://doi.org/10.1021/acssynbio.4c00578

This script is the main script for the TXTL v2 model.


In [ ]:
# Packages functions and data types --------
include("packages.jl") 

# Sequences 
sequence_file	    = "Ecoli_K12_MG1655_geneset_cds_hpaBC_ddc.fa"    # "lacS_background.fa", "Ecoli_K12_MG1655_geneset_cds_hpaBC_ddc.fa", "Ecoli_example_sequences.fa"
hybrid_file		    = "lacS.txt"
seqs_to_use 	    = (1,2323)                                 # Range of sequences in Fasta file. Takes all sequences from sequence a to sequence b
num_background_genes = 0
heterologous_genes  = ["hpaBC","ddc"]

# initialization of data containers
scale = txtl_types.init_scale()         # struct that makes it possible to influence rates by scaling them

µ = 1                    # A growth rate (0.1 to 1.8) can be chosen and the proteins of the TXTL machinery will be tuned according to that growth rate and the processivity of the RNA polymerase

## flags for efficiency factors
positive_aminoacid	= true 
anti_shinedalgarno	= false      # If no hybridization file is available put flag to false. We apply the intaRNA program's intarna function (q: aSD, t: sequence, SeedBP: 5, out: tMinE) to generate a hybridization profile along the sequence. Link to the website: https://rna.informatik.uni-freiburg.de/IntaRNA/Input.jsp;jsessionid=2AE257E91E7276283F22CD77A5A8AA03
dynamic_efficiency	= true 
save_data           = false

flags = txtl_types.flags(positive_aminoacid, anti_shinedalgarno, dynamic_efficiency)

# Data import --------
mRNA 		= to_DataFrame(readmatrix("../data/input/mRNA_concentrations.xlsx", "mRNA_concentration", "A1:B2324"))
mRNA.mRNA 	= Float64.(mRNA.mRNA);

background_genes = String.(first(mRNA.gene, num_background_genes))
genes = vcat(background_genes, heterologous_genes)
println("Number of sequences: $(length(genes))") 

mRNA = subset(mRNA, :gene => ByRow(in(genes)))


## Import information about model sp ecies and sequences 
seqs, seq_names, hyb_file_dict, species_and_seqs, idx = species_and_sequences(genes, µ, sequence_file, hybrid_file, flags, seqs_to_use)
# Are mRNA gene names in E_coli gene 


## Calculate efficiency factors 
efac_pos_aa_dict, efac_int_sd_dict, dynamic_efficiencies = efficiency_factor_initialization(seqs, seq_names, hyb_file_dict, species_and_seqs)

## calculation of initial concentrations
y0, initial_concentrations = calculation_of_initial_concentration(species_and_seqs, seqs, seq_names, scale, idx);

## calculate and load kinetic parameters
p_met, p_seq, p_dyn = parameters(initial_concentrations, species_and_seqs);

## Release factor balance
RF_balance!(y0, initial_concentrations)

# Preintegration
## Definition of mass matrix
conserved = [idx.EFG, idx.EFD] # EFG, EFTu_GDP - Conserved species, Algebraic equations are given by mass balances

M = Matrix{Float64}(I, length(y0), length(y0)) # Creates unity matrix in the size of y0

M[conserved, :] .= 0; # setting mass matrix to zero to  remove time derivatives from those variables. This converts these differential equations into algebraic constraints


tags_pre = txtl_types.data_struct_preintegration(scale, idx, species_and_seqs, flags, dynamic_efficiencies, initial_concentrations, p_met, p_seq, p_dyn, mRNA.mRNA[1])

## Integration of the DAE system without DNA in the system 
y_steady_state = preintegration(tags_pre, y0, M);


# Elongation factor balance 
EF_balance!(tags_pre, y0, y_steady_state)


# Build states vector for multiple sequences 
y0_all = combined_states_vector(tags_pre, y_steady_state, seqs, seq_names)

# Prepare per-sequence tags (for use in ODE)
local_tags_list = generate_local_tags(seqs, seq_names, tags_pre, efac_pos_aa_dict, efac_int_sd_dict, mRNA);

# Integration
# PERFORMANCE FIX: Use Diagonal mass matrix instead of dense matrix
# For 100 sequences with ~22k states, dense matrix would be ~4GB and cause O(n³) scaling
n_all = length(y0_all)
mass_diag_all = ones(Float64, n_all)
mass_diag_all[conserved] .= 0.0
M = Diagonal(mass_diag_all)  # Much more memory-efficient: O(n) instead of O(n²)

tspan 	= (0.0, 100.0) ## Define time span for the simulation

p       = generate_p(local_tags_list)

f 		= ODEFunction(ODE_parallel_distributed!, mass_matrix = M) ## Define ODEFunction to make use of mass matrix

prob 	= ODEProblem(f, y0_all, tspan, p); ## Create an ODE problem

# --- Timing the integration step ---
println("\n⏳ Integrating ODE system (distributed, test set)...")

sol     = solve(prob, FBDF(autodiff=false), reltol = 1e-8, abstol = 1e-6); ## Solve ODE system

println("Integration completed!")
# save_results(sol, local_tags_list, save_data)


🔧 Julia started with 8 threads
Number of sequences: 2



⏳ Integrating ODE system (distributed, test set)...
Integration completed!


In [10]:
# Inputs to test functions
du = similar(y0_all)
du = fill!(du, 0.0);
u = sol[end];
t = tspan;

@benchmark ODE_parallel_distributed!(du, u, p, t)


BenchmarkTools.Trial: 9018 samples with 1 evaluation per sample.
 Range (min … max):  314.700 μs …  46.265 ms  ┊ GC (min … max):  0.00% … 97.90%
 Time  (median):     399.900 μs               ┊ GC (median):     0.00%
 Time  (mean ± σ):   548.075 μs ± 743.375 μs  ┊ GC (mean ± σ):  20.45% ± 17.08%

  ▄█▇▅▄▄▃▃▂▁                                       ▂▁           ▂
  ███████████▇▇▆▇▆▅▅▃▄▃▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▅████▇▆▆▇▇▅▇▇▇ █
  315 μs        Histogram: log(frequency) by time       2.88 ms <

 Memory estimate: 2.94 MiB, allocs estimate: 825.

In [ ]:

# Vectors and numbers 
@unpack y_codons_lengths, offsets, n_metab, n_seq, du_metab_all, du_codons, y_codons, results, du_seq, du_metab = p.vectors

# Generate parts of u that need to be assembled specific to the sequence   
@views u_metab =  u[1:n_metab]				                            # Save all pool species from u in a vector 

i = 1

local_tags = p.local_tags_list[i]		                            # Extraction of local tags
idx = n_metab + offsets[i]                                          # Create index to access u on the sequence specific indexes
len = y_codons_lengths[i]            

# thread-safe local views
u_codons = @view u[idx+1 : idx+len]

# assemble without allocation
u_seq = similar(u, n_metab + len)
@views u_seq[1:n_metab] .= u_metab
@views u_seq[n_metab+1:end] .= u_codons

du_seq_local = du_seq[i]					                        # Initialization of sequence specific rates vector

du = du_seq_local
fill!(du, 0.0)
u = u_seq
p = local_tags

# @benchmark solve(prob, FBDF(autodiff=false), reltol = 1e-8, abstol = 1e-6)

@benchmark rhs_metabolites!(du, u, p)


BenchmarkTools.Trial: 10000 samples with 10 evaluations per sample.
 Range (min … max):  1.380 μs …   8.570 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     2.030 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   2.109 μs ± 428.655 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

           ▁▂▂▄█▅▅▇▃▁▁                                         
  ▁▁▁▂▂▂▄▄▆███████████▆▅▄▃▃▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▃
  1.38 μs         Histogram: frequency by time        3.92 μs <

 Memory estimate: 2.06 KiB, allocs estimate: 49.

In [13]:
@benchmark rhs_sequence!(du, u, p)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  22.000 μs …  36.175 ms  ┊ GC (min … max):  0.00% … 99.75%
 Time  (median):     44.900 μs               ┊ GC (median):     0.00%
 Time  (mean ± σ):   47.354 μs ± 443.068 μs  ┊ GC (mean ± σ):  13.02% ±  1.41%

     ▂▅█▇▅                  ▁▅▄▄▂▁                              
  ▁▃▇█████▆▃▂▂▂▂▂▃▄▃▃▂▂▃▃▄▅████████▆▅▅▄▃▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁ ▃
  22 μs           Histogram: frequency by time         76.4 μs <

 Memory estimate: 95.64 KiB, allocs estimate: 42.

In [15]:
@benchmark sol = solve(prob, FBDF(autodiff=false), reltol = 1e-8, abstol = 1e-6)

BenchmarkTools.Trial: 2 samples with 1 evaluation per sample.
 Range (min … max):  2.718 s …   2.808 s  ┊ GC (min … max): 11.50% … 12.45%
 Time  (median):     2.763 s              ┊ GC (median):    11.98%
 Time  (mean ± σ):   2.763 s ± 64.101 ms  ┊ GC (mean ± σ):  11.98% ±  0.67%

  █                                                       █  
  █▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  2.72 s         Histogram: frequency by time        2.81 s <

 Memory estimate: 5.75 GiB, allocs estimate: 1179107.